# 🌈 Lab 5C — GPUs Beyond AI: A Ray Tracer
**Foothold · GPU Computing for AI · Unit 5 (closing lesson) · Book: Step 5, Section 5.10**

Everything in this course (thousands of threads, **one thread per output**, fed by a fast memory) is not special to AI.
Today you use the same pattern to **draw a 3-D picture**: one thread per pixel, each sending a ray of light into a scene of spheres.
Then you meet the same pattern in finance (Monte Carlo).

| Part | What you do | When |
|---|---|---|
| A | Trace one pixel in Python, then every pixel on the CPU and on the GPU; time them | In class (≈ 15 min) |
| B | Shadows, a moving light, and frames per second | At home (≈ 15 min) |
| C | The same pattern in finance: a Monte Carlo estimate of π | At home (≈ 15 min) |

📘 Textbook: Kirk & Hwu Ch 2–3 (kernels and threads, again). Course site: https://aishugithub.github.io/course_GPU/

💬 **Every line of code has a comment** (after `#` in Python, after `//` inside the CUDA C kernel). The code is the same as in the book, Section 5.10.

## ▶️ How to run this notebook

1. **Save your own copy first:** menu **File → Save a copy in Drive**. Work in that copy — the original is read-only, and your copy is what you submit.
2. **Run cells in order, top to bottom.** Click a grey code cell and press **Shift + Enter** (or the ▶ button on its left). Wait for it to finish (a ✓ and the output appear) before running the next one.
3. **Switch on the GPU:** menu **Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save**. (This lab needs it. If Colab says no GPU is available, wait a while and try again — the free tier has limits.)
4. **Lost or confused?** Menu **Runtime → Restart session and run all** starts everything again from a clean slate.
5. Cells marked **✏️ YOUR TURN** are exercises: change the code, run it again, and write your answer in the text cell below it (double-click a text cell to edit it).

## 0 · Setup — run both cells

In [ ]:
import time, subprocess            # time: stopwatch; subprocess: run system commands
import numpy as np                  # arrays and maths on the CPU
import matplotlib.pyplot as plt     # charts
import torch                        # PyTorch: runs maths on the CPU or the GPU
from torch import nn                # nn: PyTorch's layers and losses

if torch.cuda.is_available():       # is there an NVIDIA GPU we can use? ("cuda" = NVIDIA's GPU system)
    device = "cuda"                 # yes: from now on, device = "cuda" means "put it on the GPU"
    print("✅ GPU found:", torch.cuda.get_device_name(0))   # the name of GPU number 0 (the first one)
else:                               # no GPU in this session
    device = "cpu"                  # fall back to the CPU
    print("⚠️ NO GPU! Go to Runtime → Change runtime type → T4 GPU → Save, then run this cell again.")   # how to get one
    print("   (Most of this notebook needs the GPU, so switch it on before going further.)")              # and why it matters

def sync():                         # GPU work runs in the background: wait for it to really finish before reading the clock
    if device == "cuda":            # (only needed when there is a GPU)
        torch.cuda.synchronize()    # wait until every queued GPU job is done

def timeit(fn, repeats=5):          # run fn several times and return the average time of one run, in seconds
    fn(); sync()                    # warm-up run (the first GPU call has extra start-up cost)
    t0 = time.perf_counter()        # start the stopwatch
    for _ in range(repeats):        # run it several times
        fn()                        # the work itself
    sync()                          # wait until the GPU has really finished all the runs
    return (time.perf_counter() - t0) / repeats   # total time ÷ number of runs

In [ ]:
# CuPy lets Python compile and run small CUDA C kernels. It comes pre-installed on Colab.
try:                                # "try": attempt this; if it fails, jump to "except" instead of crashing
    import cupy as cp               # CuPy: NumPy-like arrays on the GPU, plus our own CUDA kernels
    HAVE_CUPY = device == "cuda"    # we can only use it if there is also a GPU
    print("CuPy", cp.__version__, "ready" if HAVE_CUPY else "(installed, but no GPU in this session)")   # say whether it's usable
except ImportError:                 # CuPy is not installed
    HAVE_CUPY = False               # remember that, so later cells can say so
    print("CuPy not found. On Colab with a GPU, run:  !pip install cupy-cuda12x   then re-run this cell.")   # how to install it

def cuda_time_ms(launch, repeats=5):                # time GPU work with the GPU's own clock (CUDA events); returns ms per run
    launch(); cp.cuda.Device().synchronize()        # warm-up run, then wait for it to finish
    start, end = cp.cuda.Event(), cp.cuda.Event()   # two time-stamps on the GPU
    start.record()                                  # stamp "start" in the GPU's queue
    for _ in range(repeats):                        # several runs
        launch()                                    # queue the kernel once more
    end.record(); end.synchronize()                 # stamp "end", and wait until the GPU reaches it
    return cp.cuda.get_elapsed_time(start, end) / repeats   # time between the stamps ÷ runs

---
## Part A · One Ray per Pixel  *(Unit 5.10)*

**How a ray tracer works.** The camera sits at the origin and looks into the screen. For each pixel:
1. Send a ray from the camera through that pixel.
2. Find the **nearest sphere** the ray hits (or none: then the pixel shows the sky).
3. Colour the hit point by how directly it faces the light.

Each pixel needs only the scene, never another pixel. **Every output is independent**, just like the cells of a matrix multiply in Unit 2.3.

### A1 · The scene
Each sphere is 7 numbers: its centre (x, y, z), its radius, and its colour (red, green, blue, each 0 to 1). The "floor" is a giant sphere.

In [ ]:
SPHERES = [   # one row per sphere: centre x, y, z, radius, red, green, blue
    [ 0.0, -1001.0, -3.0, 1000.0, 0.8, 0.8, 0.8],   # the floor: a huge grey sphere whose top is at y = −1
    [ 0.0,    -0.2, -3.5,    0.8, 0.9, 0.2, 0.2],   # a red sphere in the middle
    [-1.7,    -0.4, -4.0,    0.6, 0.2, 0.8, 0.3],   # a green sphere on the left
    [ 1.6,    -0.5, -3.0,    0.5, 0.2, 0.4, 0.9],   # a blue sphere on the right
]
LIGHT = np.array([0.5, 1.0, 0.6])   # the direction pointing TOWARDS the light (up, right, towards us)
LIGHT = LIGHT / np.linalg.norm(LIGHT)   # make it length 1 (a "unit vector"), so dot products give cosines
FOV = 0.6   # how wide the camera sees (bigger = wider view)
print(len(SPHERES), "spheres; light direction", LIGHT.round(3))   # check the scene

### A2 · Trace ONE pixel, in plain Python
This function is the whole idea. Read it slowly: the GPU kernel in A4 does exactly these steps, line for line.

In [ ]:
import math   # sqrt for plain Python numbers

def trace_pixel(col, row, W, H, shadows=False):   # the colour (r, g, b) of pixel (col, row) in a W × H image
    dx = (2 * (col + 0.5) / W - 1) * (W / H) * FOV   # ray direction, x part: −1 at the left edge … +1 at the right (stretched for the image's shape)
    dy = (1 - 2 * (row + 0.5) / H) * FOV   # ray direction, y part: +1 at the top … −1 at the bottom
    dz = -1.0   # ray direction, z part: straight into the screen
    n = math.sqrt(dx*dx + dy*dy + dz*dz)   # the length of that direction
    dx, dy, dz = dx/n, dy/n, dz/n   # make it length 1

    best_t, hit = 1e30, -1   # nearest hit so far: none yet (distance "infinity", sphere −1)
    for s, (cx, cy, cz, r, *_) in enumerate(SPHERES):   # test the ray against every sphere
        ox, oy, oz = -cx, -cy, -cz   # from the sphere's centre to the camera (the camera is at 0, 0, 0)
        b = ox*dx + oy*dy + oz*dz   # how far along the ray the centre lies (a dot product)
        c = ox*ox + oy*oy + oz*oz - r*r   # squared distance to the centre, minus radius squared
        disc = b*b - c   # positive means the ray passes through the sphere
        if disc > 0:   # it does hit
            t = -b - math.sqrt(disc)   # the distance to the near side of the sphere
            if 1e-3 < t < best_t:   # in front of the camera, and nearer than anything so far?
                best_t, hit = t, s   # remember it

    if hit < 0:   # the ray hit nothing: show the sky
        k = 0.5 * (dy + 1)   # 0 looking down … 1 looking up
        return (1 - 0.5*k, 1 - 0.3*k, 1.0)   # white near the horizon, blue higher up

    cx, cy, cz, r, red, green, blue = SPHERES[hit]   # the sphere we hit
    px, py, pz = best_t*dx, best_t*dy, best_t*dz   # the hit point (camera + distance × direction)
    nx, ny, nz = (px-cx)/r, (py-cy)/r, (pz-cz)/r   # the surface's outward direction there (the "normal", length 1)
    diff = max(nx*LIGHT[0] + ny*LIGHT[1] + nz*LIGHT[2], 0.0)   # how directly the surface faces the light (0 = side-on or away)

    if shadows and diff > 0:   # is something blocking the light? (Part B)
        qx, qy, qz = px + 1e-3*nx, py + 1e-3*ny, pz + 1e-3*nz   # start just above the surface, so we don't hit ourselves
        for (cx2, cy2, cz2, r2, *_) in SPHERES:   # test a second ray, towards the light, against every sphere
            ox, oy, oz = qx-cx2, qy-cy2, qz-cz2   # from that sphere's centre to the start point
            b = ox*LIGHT[0] + oy*LIGHT[1] + oz*LIGHT[2]   # same maths as before, with the light direction
            c = ox*ox + oy*oy + oz*oz - r2*r2   # squared distance minus radius squared
            disc = b*b - c   # positive = the shadow ray passes through this sphere
            if disc > 0 and -b - math.sqrt(disc) > 1e-3:   # and the sphere is in front of us, towards the light
                diff = 0.0   # blocked: in shadow
                break   # no need to test more spheres

    light = 0.15 + 0.85 * diff   # 15% "ambient" light everywhere, plus up to 85% direct light
    return (red*light, green*light, blue*light)   # the sphere's colour, lit

print("centre pixel of a 160 × 90 image:", [round(float(v), 3) for v in trace_pixel(80, 45, 160, 90)])   # the middle pixel hits the red sphere

Now call it for **every** pixel of a small 160 × 90 image, one after another, like one CPU worker would.

In [ ]:
def render_python(W, H, shadows=False):   # the whole image, one pixel at a time
    img = np.zeros((H, W, 3), dtype=np.float32)   # an empty H × W picture with 3 colours per pixel
    for row in range(H):   # every row …
        for col in range(W):   # … and every column
            img[row, col] = trace_pixel(col, row, W, H, shadows)   # one pixel, one call
    return img   # the finished picture

t0 = time.perf_counter()   # start the stopwatch
small = render_python(160, 90)   # 14,400 pixels, one by one
t_py = time.perf_counter() - t0   # how long it took
print(f"Plain Python: {160*90:,} pixels in {t_py:.2f} s  → {t_py/(160*90)*1e6:.1f} µs per pixel")   # time per pixel
print(f"A full-HD frame (1920 × 1080) would take about {t_py/(160*90)*1920*1080:.0f} s this way")   # extrapolate to 20 lakh pixels
plt.figure(figsize=(5, 3)); plt.imshow(small); plt.axis("off")   # show the picture, without axes
plt.title("160 × 90, plain Python"); plt.show()   # title, then draw it

### A3 · Every pixel at once on the CPU (NumPy)
A fair race needs a fast CPU version. NumPy runs the **same maths on whole arrays** (one number per pixel), using compiled code and the CPU's SIMD units (Unit 2).

⚠️ **Be honest about the race:** NumPy is not the fastest possible CPU version. It writes every in-between result to memory as a big array (20 lakh numbers each, for full HD), so it is **memory-bound**, while each GPU thread keeps its numbers in registers (Unit 3.4) and is **compute-bound**. A hand-written multi-core C version would close part of the gap. The GPU still wins by a wide margin.

In [ ]:
def render_numpy(W, H, shadows=False):   # the same ray tracer, all pixels at once on the CPU
    cols, rows = np.meshgrid(np.arange(W), np.arange(H))   # two H × W arrays: every pixel's column and row
    dx = (2 * (cols + 0.5) / W - 1) * (W / H) * FOV   # ray direction, x part, for every pixel at once
    dy = (1 - 2 * (rows + 0.5) / H) * FOV   # y part
    dz = -np.ones_like(dx)   # z part: −1 everywhere
    n = np.sqrt(dx*dx + dy*dy + dz*dz)   # lengths
    dx, dy, dz = dx/n, dy/n, dz/n   # make every direction length 1

    best_t = np.full((H, W), 1e30)   # nearest hit per pixel: none yet
    hit = np.full((H, W), -1)   # which sphere: none yet
    for s, (cx, cy, cz, r, *_) in enumerate(SPHERES):   # one sphere at a time, all pixels together
        b = -cx*dx - cy*dy - cz*dz   # (camera − centre) · direction
        c = cx*cx + cy*cy + cz*cz - r*r   # one number: it is the same for every pixel
        disc = b*b - c   # positive where the ray hits this sphere
        t = -b - np.sqrt(np.maximum(disc, 0))   # distance to the near side (sqrt of 0 where it misses)
        closer = (disc > 0) & (t > 1e-3) & (t < best_t)   # pixels where this sphere is a new nearest hit
        best_t[closer], hit[closer] = t[closer], s   # remember them

    img = np.zeros((H, W, 3))   # the picture
    k = 0.5 * (dy + 1)   # sky blend for every pixel
    sky = hit < 0   # pixels that hit nothing
    img[sky] = np.stack([1 - 0.5*k, 1 - 0.3*k, np.ones_like(k)], axis=-1)[sky]   # sky colour where nothing was hit

    S = np.array(SPHERES)   # the scene as an array, so we can look rows up by hit index
    h = np.maximum(hit, 0)   # hit index, with −1 replaced by 0 (those pixels are ignored anyway)
    px, py, pz = best_t*dx, best_t*dy, best_t*dz   # hit points
    nx, ny, nz = (px-S[h,0])/S[h,3], (py-S[h,1])/S[h,3], (pz-S[h,2])/S[h,3]   # normals
    diff = np.maximum(nx*LIGHT[0] + ny*LIGHT[1] + nz*LIGHT[2], 0)   # facing the light?

    if shadows:   # shadow rays for every pixel at once
        qx, qy, qz = px + 1e-3*nx, py + 1e-3*ny, pz + 1e-3*nz   # start just above the surface
        for (cx2, cy2, cz2, r2, *_) in SPHERES:   # test every sphere
            ox, oy, oz = qx-cx2, qy-cy2, qz-cz2   # centre → start point
            b = ox*LIGHT[0] + oy*LIGHT[1] + oz*LIGHT[2]   # along the light direction
            c = ox*ox + oy*oy + oz*oz - r2*r2   # squared distance minus radius squared
            disc = b*b - c   # positive = passes through
            blocked = (disc > 0) & (-b - np.sqrt(np.maximum(disc, 0)) > 1e-3)   # sphere in front, towards the light
            diff = np.where(blocked, 0, diff)   # in shadow: no direct light

    light = 0.15 + 0.85 * diff   # ambient + direct light
    lit = S[h, 4:7] * light[..., None]   # sphere colour × light, for every pixel
    img[~sky] = lit[~sky]   # use it where a sphere was hit
    return img.astype(np.float32)   # float32, like the GPU's picture

check = render_numpy(160, 90)   # the same small picture
print("NumPy matches plain Python:", np.allclose(check, small, atol=1e-4))   # same maths → same picture

### A4 · Every pixel at once on the GPU: one thread per pixel
Here is the kernel. Compare it with `trace_pixel` above: **it is the same function**, written in CUDA C. The only new lines are the first ones, where each thread works out *which pixel is mine* (the 2-D grid from Unit 4.3).

In [ ]:
render_src = r'''
extern "C" __global__   // a kernel: runs on the GPU, launched from the CPU
void render(float *img, int W, int H,   // the output picture (H x W x 3 floats) and its size
            const float *sph, int nsph,   // the spheres (7 floats each) and how many there are
            float lx, float ly, float lz, int shadows)   // the light direction, and shadows on (1) or off (0)
{
    int col = blockIdx.x * blockDim.x + threadIdx.x;   // MY column (x direction), from my block and thread number
    int row = blockIdx.y * blockDim.y + threadIdx.y;   // MY row (y direction)
    if (col >= W || row >= H) return;   // the safety check: threads outside the picture do nothing

    float dx = (2.0f * (col + 0.5f) / W - 1.0f) * ((float)W / H) * 0.6f;   // ray direction, x part (0.6 = FOV)
    float dy = (1.0f - 2.0f * (row + 0.5f) / H) * 0.6f;   // ray direction, y part
    float dz = -1.0f;   // ray direction, z part: into the screen
    float n = sqrtf(dx*dx + dy*dy + dz*dz);   // its length
    dx /= n; dy /= n; dz /= n;   // make it length 1

    float best_t = 1e30f; int hit = -1;   // nearest hit so far: none
    for (int s = 0; s < nsph; s++) {   // test every sphere
        const float *S = sph + 7 * s;   // this sphere's 7 numbers
        float ox = -S[0], oy = -S[1], oz = -S[2];   // centre to camera (camera at 0, 0, 0)
        float b = ox*dx + oy*dy + oz*dz;   // how far along the ray the centre lies
        float c = ox*ox + oy*oy + oz*oz - S[3]*S[3];   // squared distance minus radius squared
        float disc = b*b - c;   // positive = the ray hits
        if (disc > 0.0f) {   // it hits
            float t = -b - sqrtf(disc);   // distance to the near side
            if (t > 1e-3f && t < best_t) { best_t = t; hit = s; }   // nearer than before? remember it
        }
    }

    float r, g, bl;   // this pixel's colour
    if (hit < 0) {   // hit nothing: sky
        float k = 0.5f * (dy + 1.0f);   // 0 looking down ... 1 looking up
        r = 1.0f - 0.5f*k; g = 1.0f - 0.3f*k; bl = 1.0f;   // white to blue
    } else {   // hit a sphere
        const float *S = sph + 7 * hit;   // its 7 numbers
        float px = best_t*dx, py = best_t*dy, pz = best_t*dz;   // the hit point
        float nx = (px-S[0])/S[3], ny = (py-S[1])/S[3], nz = (pz-S[2])/S[3];   // the normal
        float diff = fmaxf(nx*lx + ny*ly + nz*lz, 0.0f);   // facing the light?
        if (shadows && diff > 0.0f) {   // shadow test (Part B)
            float qx = px + 1e-3f*nx, qy = py + 1e-3f*ny, qz = pz + 1e-3f*nz;   // just above the surface
            for (int s = 0; s < nsph; s++) {   // every sphere
                const float *Q = sph + 7 * s;   // its numbers
                float ox2 = qx-Q[0], oy2 = qy-Q[1], oz2 = qz-Q[2];   // centre to start point
                float b2 = ox2*lx + oy2*ly + oz2*lz;   // along the light direction
                float c2 = ox2*ox2 + oy2*oy2 + oz2*oz2 - Q[3]*Q[3];   // squared distance minus radius squared
                float d2 = b2*b2 - c2;   // positive = passes through
                if (d2 > 0.0f && -b2 - sqrtf(d2) > 1e-3f) { diff = 0.0f; break; }   // blocked: shadow
            }
        }
        float light = 0.15f + 0.85f * diff;   // ambient + direct
        r = S[4]*light; g = S[5]*light; bl = S[6]*light;   // the sphere's colour, lit
    }
    int k = 3 * (row * W + col);   // where my pixel starts in memory (row-major, 3 floats per pixel)
    img[k] = r; img[k+1] = g; img[k+2] = bl;   // write MY pixel, and only mine
}
'''
render_k = cp.RawKernel(render_src, "render")   # compile the kernel

d_sph = cp.asarray(SPHERES, dtype=cp.float32)   # the scene, copied to the GPU once (re-run this cell if you change SPHERES)

def render_gpu(W, H, shadows=False, light=None):   # launch one thread per pixel; the picture stays in GPU memory
    L = LIGHT if light is None else light   # the light direction to use
    d_img = cp.empty((H, W, 3), dtype=cp.float32)   # space for the picture in GPU memory
    block = (16, 16)   # blocks of 16 × 16 = 256 threads
    grid = ((W + 15) // 16, (H + 15) // 16)   # enough blocks across and down (rounded up)
    render_k(grid, block, (d_img, cp.int32(W), cp.int32(H), d_sph, cp.int32(len(d_sph)), cp.float32(L[0]), cp.float32(L[1]), cp.float32(L[2]), cp.int32(shadows)))   # launch!
    return d_img   # the picture, still on the GPU

gpu_small = cp.asnumpy(render_gpu(160, 90))   # render the small picture and copy it back to compare
same = (np.abs(gpu_small - check).max(axis=2) < 1e-3).mean()   # the share of pixels whose colour matches the CPU's
print(f"GPU matches NumPy on {same:.2%} of pixels")   # float32 on the GPU vs float64 on the CPU: at most a few edge pixels differ

### A5 · The race: full-size pictures
Time the CPU (NumPy) and the GPU kernel at four sizes. The GPU time does not include copying the picture back to the CPU: a real renderer keeps the picture on the GPU and sends it straight to the screen.

In [ ]:
sizes = [(640, 360), (1280, 720), (1920, 1080), (3840, 2160)]   # small, HD, full HD, 4K
race = []   # (pixels, CPU seconds, GPU seconds) for each size

def best_ms(launch, tries=10):   # a GPU's clock speeds up and slows down from moment to moment, so time it several times...
    return min(cuda_time_ms(launch, repeats=20) for _ in range(tries))   # ...and keep the best (each try: the average of 20 runs, in ms)

def cpu_s(W, H):   # the CPU version's time in seconds: the better of two runs
    times = []   # one time per run
    for _ in range(2):   # two runs
        t0 = time.perf_counter(); render_numpy(W, H); times.append(time.perf_counter() - t0)   # time one run
    return min(times)   # keep the better one

cuda_time_ms(lambda: render_gpu(3840, 2160), repeats=500)   # warm-up: about half a second of work, so the GPU reaches full clock speed
gpu_s = [best_ms(lambda: render_gpu(W, H)) / 1000 for W, H in sizes]   # all the GPU timings first, back to back, in seconds (an idle GPU slows down)
print(f"{'size':>11} {'pixels':>10} {'CPU (NumPy)':>12} {'GPU kernel':>11} {'speed-up':>9} {'GPU frames/s':>13}")   # table header
for (W, H), t_gpu in zip(sizes, gpu_s):   # each size in turn, with its GPU time
    t_cpu = cpu_s(W, H)   # the CPU's time, in seconds
    race.append((W * H, t_cpu, t_gpu))   # keep the numbers
    print(f"{W:5d}×{H:<5d} {W*H:10,d} {t_cpu*1000:9.0f} ms {t_gpu*1000:8.2f} ms {t_cpu/t_gpu:8.0f}× {1/t_gpu:12.0f}")   # one row
big = cp.asnumpy(render_gpu(1920, 1080))   # one full-HD picture, copied back to look at
plt.figure(figsize=(8, 4.5)); plt.imshow(big); plt.axis("off")   # show the full-HD picture
plt.title("1920 × 1080, one GPU thread per pixel"); plt.show()   # title, then draw it

### ✏️ YOUR TURN (A)
1. Fill in the table from your run.

| Size | CPU (NumPy) | GPU kernel | Speed-up | GPU frames per second |
|---|---|---|---|---|
| 640 × 360 | | | | |
| 1920 × 1080 | | | | |
| 3840 × 2160 | | | | |

2. Games need 60 frames per second. Which sizes reach 60 on the GPU? On the CPU?
3. In the kernel, what is **one output**, and why are the outputs independent? Name the line that makes each thread pick its own pixel.
4. Run the race cell twice. Why do the smallest pictures' GPU times jump about from run to run? (Hint: Unit 4.4, launch overhead.)

*Your answers:*

---
## Part B · Shadows and a Moving Light  *(at home)*

**Shadows** cost a second ray per lit pixel (towards the light), so each thread does more work. How much slower is it? Then the cell swings the light once round the scene in 60 frames and measures the frames per second.

In [ ]:
W, H = 1920, 1080   # full HD
cuda_time_ms(lambda: render_gpu(3840, 2160), repeats=500)   # warm-up again: the GPU slowed down while the CPU was racing
offs, ons = [], []   # timings without and with shadows
for _ in range(20):   # 20 tries, taking turns, so both see the same clock speed
    offs.append(cuda_time_ms(lambda: render_gpu(3840, 2160, shadows=False), repeats=20))   # ms per 4K frame without shadows (4K: big enough that the ray work, not the launch, dominates)
    ons.append(cuda_time_ms(lambda: render_gpu(3840, 2160, shadows=True), repeats=20))   # ms per 4K frame with shadows
t_off, t_on = min(offs), min(ons)   # keep the best of each
print(f"4K frame, no shadows: {t_off:.2f} ms   with shadows: {t_on:.2f} ms   ({t_on/t_off:.1f}× the work time)")   # compare
pic = cp.asnumpy(render_gpu(W, H, shadows=True))   # one picture with shadows

frames, shown = 60, []   # 60 frames; keep a few to look at
t0 = time.perf_counter()   # start the stopwatch
for f in range(frames):   # one frame at a time
    a = 2 * np.pi * f / frames   # the light's angle this frame (one full turn in 60 frames)
    L = np.array([np.cos(a), 1.0, np.sin(a)]); L = L / np.linalg.norm(L)   # the light direction, length 1
    d = render_gpu(W, H, shadows=True, light=L)   # render this frame on the GPU
    if f % 15 == 0: shown.append(cp.asnumpy(d))   # every 15th frame: copy back to look at
cp.cuda.Device().synchronize()   # wait until the GPU has finished every frame
fps = frames / (time.perf_counter() - t0)   # frames per second, including the few copies
print(f"{frames} full-HD frames with shadows: {fps:.0f} frames per second")   # the result
plt.figure(figsize=(8, 4.5)); plt.imshow(pic); plt.axis("off")   # the picture with shadows
plt.title("With shadows: a second ray per pixel"); plt.show()   # title, then draw it
fig, ax = plt.subplots(1, 4, figsize=(14, 2.3))   # four small pictures side by side
for i, im in enumerate(shown):   # each kept frame of the moving light
    ax[i].imshow(im); ax[i].axis("off"); ax[i].set_title(f"frame {15*i}")   # show it with its frame number
plt.show()   # draw them

### ✏️ YOUR TURN (B)
1. How much longer did a frame take with shadows? Is that close to "one extra ray per pixel"?
2. Add a fifth sphere: put a new row in `SPHERES` (re-run A1 and the kernel cell in A4, then this part). Try `[0.9, 0.6, -5.0, 0.7, 0.9, 0.8, 0.2]` (a yellow sphere behind). Does the frame rate change much? Why does each extra sphere cost every pixel a little?
3. Real games test rays against millions of triangles, not four spheres. Which GPU unit did NVIDIA add for exactly this test? (Unit 5.10)

*Your answers:*

---
## Part C · The Same Pattern in Finance: Monte Carlo  *(at home)*

Banks price options by simulating lakhs of **independent** random futures and averaging them. The simplest Monte Carlo job: throw random darts at a 1 × 1 square. The fraction landing inside the quarter circle is π/4, so **π ≈ 4 × (darts inside) ÷ (all darts)**. One dart = one independent output.

In [ ]:
N = 50_000_000   # 5 crore darts

def pi_cpu():   # on the CPU, with NumPy
    x = np.random.rand(N).astype(np.float32)   # N random x positions between 0 and 1
    y = np.random.rand(N).astype(np.float32)   # N random y positions
    return 4 * np.count_nonzero(x*x + y*y < 1) / N   # 4 × fraction inside the circle

def pi_gpu():   # the same on the GPU, with CuPy (one GPU thread per dart, inside CuPy)
    x = cp.random.rand(N, dtype=cp.float32)   # N random x positions, made in GPU memory
    y = cp.random.rand(N, dtype=cp.float32)   # N random y positions
    return 4 * int(cp.count_nonzero(x*x + y*y < 1)) / N   # count on the GPU; only one number comes back

t0 = time.perf_counter(); p_cpu = pi_cpu(); t_cpu = time.perf_counter() - t0   # time the CPU
pi_gpu(); pi_gpu()   # two warm-up runs: the first GPU call compiles CuPy's kernels (seconds)
t0 = time.perf_counter()   # start the stopwatch
for _ in range(3): p_gpu = pi_gpu()   # three timed runs (int() inside waits for each answer)
t_gpu = (time.perf_counter() - t0) / 3   # average time of one run
print(f"CPU: π ≈ {p_cpu:.5f} in {t_cpu*1000:.0f} ms")   # CPU result
print(f"GPU: π ≈ {p_gpu:.5f} in {t_gpu*1000:.1f} ms   ({t_cpu/t_gpu:.0f}× faster)")   # GPU result
print(f"true π = {np.pi:.5f}")   # for comparison

### ✏️ YOUR TURN (C)
1. Run the cell with `N = 50_000` and then `N = 50_000_000`. How does the error in π change? (Monte Carlo needs **lots** of samples, which is why it suits the GPU.)
2. Fill in the "one pattern" table:

| Field | One output is … | Why are the outputs independent? |
|---|---|---|
| AI (Unit 2) | one cell of a matrix multiply | |
| Ray tracing (Part A) | | |
| Monte Carlo (Part C) | | |
| Weather simulation | one grid cell at one time step | |

*Your answers:*

---
## 🎓 Where next?
You have now used one idea, **one thread per independent output**, for AI, graphics and finance. The same skills (counting the work, CPU or GPU, mapping outputs to threads, memory and precision, profiling) are what GPU, graphics, HPC and AI-kernel engineers use every day.

**Mini-project idea:** extend this ray tracer (reflections, more spheres, an animation), or build another GPU application (an image filter, a heat-flow simulation), and plan, measure and optimise it like any AI task. Open `MiniProject_Template.ipynb` from the course site.